# Task 10 -- OU variance-ratio floor sensitivity (`docs/open_questions.md` §6.1)

Sweeps `train.local_residual.fit_local_residual`'s `variance_floor` for one
variable (treated as `V=1`, a valid degenerate case of its `(T, N_ocean, V)`
signature) and reports the `active_mask` fraction at each floor. Full
end-to-end validation (measuring the *downstream* error the floor's
omission introduces) stays blocked on simulation infrastructure that
doesn't exist yet -- this task's scope is explicitly the floor's direct,
measurable effect, not that larger validation. See
`tasks/10-ou-variance-floor-validation.md`.

**Note:** `fit_local_residual` itself only needs plain `(T, N_ocean, V)`
arrays -- it does not call into `train.tucker`'s `TuckerFit`, so this
notebook builds `Z`/`Z_coherent` directly via `tensorly` (per-variable
design) rather than through `train.tucker.tucker_fit`/`coherent_reconstruction`,
which still assume the old joint-variable tensor shape.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2025, 12, 1, tzinfo=timezone.utc)

VARIABLE = "wsh"
X_raw = load_training_window(DATA_DIR, T_START, T_END, [VARIABLE])
X = X_raw[VARIABLE].transpose("wave_lat", "wave_lon", "time").values

ocean_mask = ~np.isnan(X).any(axis=-1)
tucker_mask = np.broadcast_to(ocean_mask[:, :, None], X.shape).astype(np.float64)
X_filled = np.where(ocean_mask[:, :, None], X, 0.0)
n_lat, n_lon, T = X_filled.shape


## Build Z (Gaussianized anomalies) and Z_coherent (Tucker+DMD deterministic reconstruction)

In [ ]:
from train.copula import GaussianCopula
from train.dmd import dmd_fit

X_anom = X_filled - X_filled.mean(axis=-1, keepdims=True)
anom_ds = xr.Dataset({VARIABLE: (("time", "wave_lat", "wave_lon"), X_anom.transpose(2, 0, 1))})
Z_full = GaussianCopula(n_bins=500).fit_transform(anom_ds)[VARIABLE].values  # (T, n_lat, n_lon)

R_LAT, R_LON = 10, 20
Z_for_tucker = Z_full.transpose(1, 2, 0)  # back to (lat, lon, time) for partial_tucker
core, factors = partial_tucker(
    np.nan_to_num(Z_for_tucker, nan=0.0), modes=[0, 1, 2], rank=[R_LAT, R_LON, T],
    init="random", tol=1e-5, n_iter_max=2000, random_state=SEED, mask=tucker_mask,
)
dt = 3.0 * 3600.0
dmd = dmd_fit(core, r_dmd=20, dt=dt, rng=rng)

# Deterministic-only reconstruction (b0 * exp(lam_cont * t), NO mode-OU wobble
# added -- local_residual.py's coherent_reconstruction docstring is explicit
# that including the wobble here would leak coherent-layer stochastic
# variance into this fit and double-count it).
tt = np.arange(T)
a_det = dmd.b0[:, None] * (dmd.Lambda[:, None] ** tt[None, :])  # (r_dmd, T)
core_det_flat = (dmd.Phi @ a_det).real  # (R_LAT*R_LON, T)
core_det = core_det_flat.T.reshape(T, R_LAT, R_LON)
Z_coherent_grid = tl.tucker_to_tensor((core_det.transpose(1, 2, 0), factors))  # (n_lat, n_lon, T)

# Flatten to (T, N_ocean, V=1) per fit_local_residual's expected shape.
Z_ocean = Z_full[:, ocean_mask][:, :, None]  # (T, N_ocean, 1)
Z_coherent_ocean = Z_coherent_grid.transpose(2, 0, 1)[:, ocean_mask][:, :, None]  # (T, N_ocean, 1)
print(f"Z_ocean {Z_ocean.shape}, Z_coherent_ocean {Z_coherent_ocean.shape}")


## Sweep `variance_floor` and report `active_mask` fraction

In [ ]:
from train.local_residual import fit_local_residual

for floor in [0.0, 0.01, 0.02, 0.05, 0.1, 0.2]:
    fit = fit_local_residual(Z_ocean, Z_coherent_ocean, dt=dt, variance_floor=floor)
    print(f"variance_floor={floor}: {fit.active_mask.mean():.1%} of cells active, "
          f"median variance_ratio={np.median(fit.variance_ratio):.4f}")


## Decision-rule diagnostic

Record the `active_mask`-fraction-vs-floor curve in
`tasks/10-ou-variance-floor-validation.md` §4, noting explicitly (per
`ornstein_uhlenbeck_processes.md` §6) that this measures the floor's
*direct* effect on how many cells get a local-residual OU process, not the
downstream reconstruction-error cost of excluding the rest -- that
validation needs simulation infrastructure this task does not build.